# Architecture 2.0: Capstone Workshop
Welcome to the Lighthouse Architecture Loop. In this tutorial, we will explore the **Scissors Gap**—the massive cost imbalance between AI generation and physical hardware verification.

### Phase 1: Setup and The Lighthouse Request
We define the strict architectural constraints for our 3W RISC-V XR SoC.

In [ ]:
import sys
from pathlib import Path
import urllib.request
import re
import yaml
import subprocess
import tempfile

# Ensure arch2_labs is in the path
sys.path.insert(0, str(Path().resolve().parent))
from arch2_labs.scale_env import run_example
from arch2_labs.decisions import record_human_decision

print("Environment initialized!")

### Phase 2A: The Mirage (LLM Generation)
We ask a local LLM to act as our architect. It will confidently generate dimensions for a systolic array.

In [ ]:
def ask_llm():
    prompt = "You are an AI architect. Propose two new dimensions (rows and cols) for a systolic array that uses exactly 1024 PEs total. Output only a strict JSON list of objects with 'array_rows' and 'array_cols'."
    data = json.dumps({"model": "gemma3:1b", "prompt": prompt, "stream": False, "format": "json"}).encode('utf-8')
    req = urllib.request.Request("http://localhost:11434/api/generate", data=data, headers={'Content-Type': 'application/json'})
    response = urllib.request.urlopen(req)
    result = json.loads(response.read())
    return json.loads(result['response'])

print("Generating candidates via Ollama...")
try:
    llm_candidates = ask_llm()
    print("LLM Output:", llm_candidates)
except Exception as e:
    print("Ollama failed, using fallbacks.")
    llm_candidates = [{'array_rows': 4, 'array_cols': 256}, {'array_rows': 64, 'array_cols': 16}]


### Phase 2B: Architectural Proxy Verification (SCALE-Sim)
Now, we run the LLM's candidates through a cycle-accurate architectural simulator. Will they pass the 90,000 cycle deadline?

In [ ]:
# Setup candidates.yaml dynamically
yaml_path = Path().resolve().parent / "examples" / "scale_proxy_mirage" / "configs" / "candidates.yaml"
with open(yaml_path, 'r') as f:
    spec = yaml.safe_load(f)

# Add baseline and LLM candidates
spec['candidates'] = [{'candidate_id': 'balanced_16x16', 'array_rows': 16, 'array_cols': 16, 'source': 'Human Baseline'}]
for i, c in enumerate(llm_candidates):
    spec['candidates'].append({'candidate_id': f'llm_gen_{i}', 'array_rows': c['array_rows'], 'array_cols': c['array_cols'], 'source': 'Ollama'})

with open(yaml_path, 'w') as f:
    yaml.dump(spec, f)

print("Running SCALE-Sim...")
out_dir = run_example()
print(f"Results written to {out_dir}")

with open(out_dir / "verification_evidence.json", 'r') as f:
    evidence = json.load(f)

survivors = []
for outcome in evidence["candidate_outcomes"]:
    status = "✅ PASSED" if outcome["accepted"] else "❌ FAILED"
    print(f"{outcome['candidate_id']}: {status}")
    if outcome["accepted"]:
        survivors.append(outcome['candidate_id'])


### Phase 2C: Physical EDA Synthesis (Yosys)
For the candidates that survived SCALE-Sim, we push them down to the physical EDA level. We synthesize a MAC unit in Yosys to determine the physical standard cell area required for the array.

In [ ]:
def run_eda_synthesis(candidate_id, rows, cols):
    eda_dir = Path(tempfile.mkdtemp(prefix="arch2_eda_"))
    rtl_path = eda_dir / f"{candidate_id}.v"
    
    rtl = """module mac (
        input clk, input [7:0] a, input [7:0] b, input [15:0] p_sum, output reg [15:0] out
    );
        always @(posedge clk) out <= p_sum + (a * b);
    endmodule"""
    rtl_path.write_text(rtl)
    
    yosys_script = eda_dir / "synth.ys"
    yosys_script.write_text(f"""read_verilog {rtl_path}\nhierarchy -check -top mac\nsynth -top mac\nabc -g gates\nstat""")
    
    result = subprocess.run(["yosys", "-s", str(yosys_script)], capture_output=True, text=True)
    
    mac_cells = 0
    for line in result.stdout.splitlines():
        if "Number of cells:" in line:
            mac_cells = int(line.split()[-1])
            break
            
    return mac_cells * (rows * cols)

if survivors:
    winner = survivors[0]
    cand_data = next(c for c in spec['candidates'] if c['candidate_id'] == winner)
    gate_count = run_eda_synthesis(winner, cand_data['array_rows'], cand_data['array_cols'])
    print(f"\n>> EDA Synthesis Results for {winner}:")
    print(f"Total Array Physical Standard Cells: {gate_count}")
else:
    print("No candidates survived to reach EDA synthesis.")


### Phase 3: The Commitment (Cryptographic Ownership)
You must own this decision. Review the machine's recommendation, document the residual risk, and sign the receipt.

In [ ]:
decision_dict = {
    "decision": "accept_baseline",
    "rationale": "The baseline balanced_16x16 passed SCALE-Sim latency constraints and EDA synthesis area checks.",
    "residual_risk": "Yosys does not perform physical routing; thermal hot-spots may still occur.",
    "would_overturn": "If OpenROAD physical design flags high congestion routing failures."
}

try:
    record_human_decision(out_dir, decision_dict)
    print("✅ Cryptographic receipt generated successfully. You own this architecture.")
except Exception as e:
    print(f"❌ Failed to seal receipt: {e}")
